# Data Preprocessing and merging

This notebook contains gene variable homologation to human genes (preserving non-orthologous NMR genes) and QC filtering.

### Pipeline

#### NMR

1) Load filtered data obtained from cellbender (add .obs[['region', 'species', 'replicate' , 'sample_id']])
2) Find mitochondrial and ribosomal genes
3) Calculate QC metrics (standard + log1p)
4) Set QC thresholds with `find_means.py` per region
5) QC Filtering
6) Map gene variables to homologus gene variables
7) Save counts layer
8) Save AnnData object per region

Available kernel & conda env: `scanpy-env`


In [2]:
!ls ../..


big_data			   miniconda3
data				   nakedmolerat-scRNAseq
filter_qc_clusters_broken_copy.py  NMR-snRNA-seq
filter_qc_clusters_copy.py	   output.png
filter_qc_clusters.py		   qc_clustering_annother_nb_more.ipynb
layouts				   scanpy-env.yml
__marimo__


In [1]:
# Revision number (change every time you run it again)
rev_n = 15
# File paths
path_to_nmr_input_dir = '../../big_data/filtered/'
path_to_human_orthology ='../data/gene_lists/nmr_human_orthology.csv'
path_to_output_dir = '../results/preprocessing/'


In [2]:
import sys
import os

# Get the path to the directory containing 'scripts/' 
# (assuming your notebook is in the project root or a sibling folder)
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
scripts_path = os.path.join(project_root, "scripts")

if scripts_path not in sys.path:
    sys.path.insert(0, scripts_path)


In [3]:
# Check that all paths are correct
import os
all_paths = [path_to_nmr_input_dir, path_to_human_orthology]
for path in all_paths:
   if os.path.exists(path):
      print(path,' exists.')
   else: print(path, 'does NOT exist. Please correct it.')

../../big_data/filtered/  exists.
../data/gene_lists/nmr_human_orthology.csv  exists.


In [4]:
import glob
import scanpy as sc
import anndata as ad
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import math

In [5]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=UserWarning)
warnings.simplefilter(action='ignore', category=RuntimeWarning)

## Naked Mole Rat Datasets

In [6]:
nmr_files = sorted(glob.glob('/home/ratopin/big_data/filtered/*_filtered.h5ad'))
nmr_files

['/home/ratopin/big_data/filtered/NMR1_cerebral_filtered.h5ad',
 '/home/ratopin/big_data/filtered/NMR2_cerebral_filtered.h5ad',
 '/home/ratopin/big_data/filtered/NMR3_hippocampus_filtered.h5ad',
 '/home/ratopin/big_data/filtered/NMR4_hippocampus_filtered.h5ad',
 '/home/ratopin/big_data/filtered/NMR5_midbrain_filtered.h5ad',
 '/home/ratopin/big_data/filtered/NMR6_midbrain_filtered.h5ad']

In [7]:
# Load gene otholog reference table (adapted with `gene_reference.ipynb`)
gene_reference = pd.read_csv(path_to_human_orthology)
gene_reference.head()

,human_gene_name,human_gene_acc,nmr_gene_name,nmr_gene_acc,perc_id,orthology_type,human_name_rep,nmr_name_rep,human_chr,nmr_chr
0,A2M,ENSG00000175899,A2m,ENSHGLG00000000871,77.7476,ortholog_one2one,1.0,1.0,12,5
1,A3GALT2,ENSG00000184389,A3GALT2,ENSHGLG00000012021,78.2353,ortholog_one2one,1.0,1.0,1,7
2,A4GALT,ENSG00000128274,A4GALT,ENSHGLG00000048655,77.0538,ortholog_one2one,1.0,1.0,22,5
3,A4GNT,ENSG00000118017,A4GNT,ENSHGLG00000014457,79.3510,ortholog_one2one,1.0,1.0,3,8
4,AAAS,ENSG00000094914,AAAS,ENSHGLG00000002843,93.0147,ortholog_one2one,2.0,1.0,12,17


In [8]:
nmr_adatas = []
nmr_gene_map = dict(zip(gene_reference['nmr_gene_acc'], gene_reference['human_gene_name']))
nmr_gene_chr_map = dict(zip(gene_reference['nmr_gene_acc'],gene_reference['nmr_chr']))

for file in nmr_files:
    # CellBender output should be read with read_10x_h5
    try: 
        adata = sc.read_10x_h5(file)
    except:
        adata = sc.read(file)

    adata.var['species'] = 'nmr'

    sample = file.replace('/home/ratopin/big_data/filtered/NMR', '').replace('_filtered.h5ad', '').split('_', 1)
    sample_id = sample[0]
    tissue = sample[1]

    print(f'Loading sample {sample_id} of {tissue}')

    adata.obs['species'] = 'nmr'
    adata.obs['tissue'] = tissue
    adata.obs['sample_id'] = sample_id
    adata.obs['replicate'] = (int(sample_id)+1)%2+1

    # Map NMR gene IDs to global names
    adata.var['genome'] = 'HetGla_female1'
    adata.var['chromosome'] = adata.var['gene_ids'].map(nmr_gene_chr_map)
    adata.var['global_name'] = adata.var['gene_ids'].map(nmr_gene_map)
    adata.var['global_name'].fillna(pd.Series(adata.var.index, index=adata.var.index), inplace=True)
    nmr_adatas.append(adata)

nmr_adatas

Loading sample 1 of cerebral
Loading sample 2 of cerebral
Loading sample 3 of hippocampus
Loading sample 4 of hippocampus
Loading sample 5 of midbrain
Loading sample 6 of midbrain


[AnnData object with n_obs × n_vars = 11353 × 20774
     obs: 'species', 'sample', 'tissue', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'log1p_total_counts_hb', 'pct_counts_hb', 'total_counts_tf', 'log1p_total_counts_tf', 'pct_counts_tf', 'n_counts', 'qc_composite', 'qc_label', 'sample_id', 'replicate'
     var: 'gene_ids', 'feature_types', 'genome', 'mt', 'ribo', 'hb', 'tf', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'species', 'chromosome', 'global_name',
 AnnData object with n_obs × n_vars = 16526 × 20774
     obs: 'species', 'sample', 'tissue', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_

In [9]:
nmr_adatas[0].var[nmr_adatas[0].var['global_name'].isna()]

,gene_ids,feature_types,genome,mt,ribo,hb,tf,n_cells_by_counts,mean_counts,log1p_mean_counts,pct_dropout_by_counts,total_counts,log1p_total_counts,species,chromosome,global_name


In [10]:
for ad in nmr_adatas:
    if 'global_name' not in ad.var.columns:
        ad.var['global_name'] = ad.var.index
    ad.var_names_make_unique()

In [11]:
samples = [os.path.basename(f).replace('_filtered.h5ad', '').split('_',1)[0] for f in nmr_files]

cortex_nmr_adata = sc.concat(nmr_adatas[0:2],axis=0, join='outer', label='sample', keys=samples[0:2], index_unique='-')
hipo_nmr_adata = sc.concat(nmr_adatas[2:4],axis=0, join='outer', label='sample', keys=samples[2:4], index_unique='-')
midbrain_nmr_adata = sc.concat(nmr_adatas[4:6],axis=0, join='outer', label='sample', keys=samples[4:6], index_unique='-')


In [12]:
cortex_nmr_adata.obs['tissue'] = "cerebral cortex"

In [13]:
nmr_datas = [cortex_nmr_adata, hipo_nmr_adata, midbrain_nmr_adata]

#### QC basic filtering

- Cerebral cortex: total counts > 200
- Hippocampus: total counts > 200
- Midbrain: total counts > 45 *This filtering is not useful for this sample anyway, so another type of filtering is needed.


In [14]:
nmr_mito_genes = open('../data/gene_lists/mito_genes.txt', 'r').read().splitlines()
human_ribo_genes = pd.read_csv('../data/gene_lists/human_ribo_genes.csv', header=1)
human_ribo_genes = human_ribo_genes['Approved symbol'].tolist()
len(human_ribo_genes)

247

In [15]:
for adata in nmr_datas:
    print(adata.obs['tissue'][0])
    # mitochondrial genes, "MT-" for human, "Mt-" for mouse
    if 'mt' not in adata.var.columns:
        try: adata.var["mt"] = adata.var['Chromosome'] == 'MT'
        except: adata.var["mt"] = adata.var_names.str.startswith(("mt-", "MT-","Mt-")) |[gene in nmr_mito_genes for gene in adata.var_names]
    print(f'Found {adata.var["mt"].sum()} mito genes.')
    # ribosomal genes
    try: adata.var["ribo"] = [name in human_ribo_genes for name in adata.var['Gene']]
    except: adata.var["ribo"] = adata.var_names.str.startswith(("RPS", "RPL", "Rpl", "Rps"))
    print(f'Found {adata.var["ribo"].sum()} ribo genes.\n')
    sc.pp.calculate_qc_metrics(adata, qc_vars=["mt", "ribo"], inplace=True)

cerebral cortex
Found 13 mito genes.
Found 78 ribo genes.

hippocampus
Found 13 mito genes.
Found 78 ribo genes.

midbrain
Found 13 mito genes.
Found 78 ribo genes.



In [16]:
min_total_counts  = [200,200,30]

In [17]:
print('Total cells:', nmr_datas[0].n_obs)
print(f'total_counts <= {min_total_counts[0]}:\n',nmr_datas[0].obs[nmr_datas[0].obs['total_counts']<=min_total_counts[0]].value_counts('sample'))
print('Remaining cells:', nmr_datas[0].n_obs- len(nmr_datas[0].obs[nmr_datas[0].obs['total_counts']<=min_total_counts[0]]))

print('\nTotal cells:', nmr_datas[1].n_obs)
print(f'total_counts <= {min_total_counts[1]}:\n',nmr_datas[1].obs[nmr_datas[1].obs['total_counts']<=min_total_counts[1]].value_counts('sample'))
print('Remaining cells:', nmr_datas[1].n_obs- len(nmr_datas[1].obs[nmr_datas[1].obs['total_counts']<=min_total_counts[1]]))

print('\nTotal cells:', nmr_datas[2].n_obs)
print(f'total_counts <= {min_total_counts[2]}:\n',nmr_datas[2].obs[nmr_datas[2].obs['total_counts']<=min_total_counts[2]].value_counts('sample'))
print('Remaining cells:', nmr_datas[2].n_obs- len(nmr_datas[2].obs[nmr_datas[2].obs['total_counts']<=min_total_counts[2]]))



Total cells: 27879
total_counts <= 200:
 sample
NMR2    2331
NMR1    1156
Name: count, dtype: int64
Remaining cells: 24392

Total cells: 14311
total_counts <= 200:
 sample
NMR4    349
NMR3    283
Name: count, dtype: int64
Remaining cells: 13679

Total cells: 9136
total_counts <= 30:
 sample
NMR5    406
NMR6    390
Name: count, dtype: int64
Remaining cells: 8340


In [18]:
from plot_functions import plot_qc_violin_with_thres


In [21]:
qc_cuts = {"n_genes_by_counts":[3,3,3], 
        "min_total_counts":[200,200,30], 
        "max_total_counts":[22000,22000,10000],
        "pct_counts_mt": [3,3,3], 
        'pct_counts_ribo': [7,7,7]}

# Map tissue types to indices
tissue_to_idx = {"cerebral_cortex": 0, "hippocampus": 1, "midbrain": 2}

# Loop through all datasets
for adata in nmr_datas:
    # 1. Determine which tissue this AnnData belongs to
    tissue = adata.obs['tissue'].iloc[0]
    idx = tissue_to_idx.get(tissue, 0)
    
    # 2. Slice the tissue-specific thresholds from your qc_cuts lists
    tissue_qc_cuts = {
        "n_genes_by_counts": qc_cuts["n_genes_by_counts"][idx],
        "min_total_counts": qc_cuts["min_total_counts"][idx],
        "max_total_counts": qc_cuts["max_total_counts"][idx],
        "pct_counts_mt": qc_cuts["pct_counts_mt"][idx],
        "pct_counts_ribo": qc_cuts["pct_counts_ribo"][idx]
    }
    
    # 3. Call the function with the correct arguments
    fig, axes = plot_qc_violin_with_thres(
        adata=adata,                 # Pass the active AnnData object
        qc_cuts=tissue_qc_cuts,      # Pass the sliced dictionary of numbers
        region_col="tissue",          # Point to the column storing the tissue name
        figsize=(22, 5)
    )
    out_folder = "../results/preprocessing/"
    os.makedirs(out_folder, exist_ok=True)
    save_path = os.path.join(out_folder, f"{tissue}_qc_violins_before_filtering.png")
    fig.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved QC violin plots for {tissue} to {out_folder}{tissue}_qc_violins_before_filtering.png")

Plotting cerebral cortex
Saved QC violin plots for cerebral cortex to ../results/preprocessing/cerebral cortex_qc_violins_before_filtering.png
Plotting hippocampus
Saved QC violin plots for hippocampus to ../results/preprocessing/hippocampus_qc_violins_before_filtering.png
Plotting midbrain
Saved QC violin plots for midbrain to ../results/preprocessing/midbrain_qc_violins_before_filtering.png


In [87]:
# Filter out low-quality cells in nmr data
filtered_nmr_datas = []
for i, adata in enumerate(nmr_datas):
    print(f'Filtering {adata.obs["tissue"].iloc[0]} NMR data:')
    nmr_data = adata.copy()
    nmr_data = nmr_data[nmr_data.obs['total_counts'] > qc_cuts['min_total_counts'][i],:]
    nmr_data = nmr_data[nmr_data.obs['total_counts'] < qc_cuts['max_total_counts'][i],:]
    nmr_data = nmr_data[nmr_data.obs['pct_counts_mt'] < qc_cuts['pct_counts_mt'][i],:]
    nmr_data = nmr_data[nmr_data.obs['pct_counts_ribo'] < qc_cuts['pct_counts_ribo'][i],:]
    sc.pp.filter_genes(nmr_data, min_cells=qc_cuts['n_genes_by_counts'][i], inplace=True)
    filtered_nmr_datas.append(nmr_data)
    print(f'Filtered {adata.obs["tissue"].iloc[0]} NMR data: {adata.n_obs} cells, {adata.n_vars} genes → {nmr_data.n_obs} cells, {nmr_data.n_vars} genes.\n')

Filtering cerebral cortex NMR data:
Filtered cerebral cortex NMR data: 27879 cells, 20774 genes → 24378 cells, 17413 genes.

Filtering hippocampus NMR data:
Filtered hippocampus NMR data: 14311 cells, 20774 genes → 13634 cells, 17592 genes.

Filtering midbrain NMR data:
Filtered midbrain NMR data: 9136 cells, 20774 genes → 8070 cells, 13915 genes.



In [88]:
## Save NMR datas
for nmr_data in filtered_nmr_datas:
    tissue = nmr_data.obs['tissue'].iloc[0].replace(" ", "_")
    filename = f'/home/ratopin/big_data/nmr/{tissue}_nmr_adata_filtered{rev_n}.h5ad'
    nmr_data.write_h5ad(filename)
    print(f"AnnData from NMR {tissue} successfully saved to {filename}")

AnnData from NMR cerebral_cortex successfully saved to /home/ratopin/big_data/nmr/cerebral_cortex_nmr_adata_filtered14.h5ad
AnnData from NMR hippocampus successfully saved to /home/ratopin/big_data/nmr/hippocampus_nmr_adata_filtered14.h5ad
AnnData from NMR midbrain successfully saved to /home/ratopin/big_data/nmr/midbrain_nmr_adata_filtered14.h5ad


## Extra

In [111]:
from scipy import sparse

def drop_lower_count_duplicates(adata, gene_key="global_name", verbose=True):
    """
    Remove duplicated genes by keeping the one with highest global expression.
    Prints info about duplicates if verbose=True.
    
    Parameters
    ----------
    adata : AnnData
    gene_key : str
        Column in `adata.var` to identify gene names (default "global_name").
    verbose : bool
        If True, print duplicate genes and their counts.
    
    Returns
    -------
    AnnData with unique genes
    """
    # Get gene names
    if gene_key in adata.var.columns:
        genes = adata.var[gene_key].astype(str)
    else:
        genes = adata.var_names.astype(str)

    # Compute total counts per gene (sparse-aware)
    if sparse.issparse(adata.X):
        total_counts = np.array(adata.X.sum(axis=0)).flatten()
    else:
        total_counts = adata.X.sum(axis=0)

    # Build dataframe
    df = pd.DataFrame({
        "gene": genes.values,
        "total_counts": total_counts,
        "idx": np.arange(len(genes))
    })

    # Find duplicates
    dupes = df[df.duplicated("gene", keep=False)].sort_values("gene")

    if verbose and not dupes.empty:
        print("\nDuplicate genes found:")
        for g, sub in dupes.groupby("gene"):
            print(f"  {g}:")
            for _, row in sub.iterrows():
                print(f"    idx={row['idx']} total_counts={row['total_counts']}")
    
    # Keep max count per gene
    keep_idx = df.loc[df.groupby("gene")["total_counts"].idxmax(), "idx"].values

    # Subset AnnData
    adata_unique = adata[:, keep_idx].copy()
    adata_unique.var_names = genes.iloc[keep_idx].values

    return adata_unique


#### Tidy up NMR


For each region of nmr adata:

1. Map genes to gene_reference['human_gene_name']

2. **Fill NaNs** with original nmr genes (preserve nmr genes for future analysis)

3. Drop duplicates (In case any gene name repeats during orthologus merging)

4. Set ['human_gene_name'] column to var_names

In [114]:
for i in range(len(filtered_nmr_datas)):
    adata = filtered_nmr_datas[i]
    print(f"{adata.obs['tissue'].iloc[0]}: {adata.n_obs} cells, {adata.n_vars} genes")
    print(adata.var_names[:5])
    adata.var['global_name'] = adata.var_names.map(dict(zip(gene_reference['nmr_gene_name'], gene_reference['human_gene_name'])))
    adata.var['original_names'] = adata.var_names.to_list()
    adata.var['global_name'].fillna(adata.var['original_names'], inplace=True)
    # Drop duplicates (keeps the higher-count gene)
    adata = drop_lower_count_duplicates(adata)
    print(f'After droping duplicates: {adata.n_obs} cells, {adata.n_vars} genes')
    # Set global_name as var_names and make unique
    adata.var_names = adata.var['global_name']
    adata.var_names_make_unique()
    # Put the modified AnnData back into the tidy list
    filtered_nmr_datas[i] = adata

cerebral cortex: 24378 cells, 17413 genes
Index(['ZMYND10', 'AMIGO3', 'TNFSF10', 'NPRL2', 'GMPPB'], dtype='object')

Duplicate genes found:
  ACP1:
    idx=10175 total_counts=168.0
    idx=14752 total_counts=158.0
  AKAP7:
    idx=5409 total_counts=70.0
    idx=5430 total_counts=1480.0
  AMD1:
    idx=14615 total_counts=32.0
    idx=16699 total_counts=2714.0
  AMN:
    idx=1703 total_counts=215.0
    idx=17171 total_counts=307.0
  APPL2:
    idx=12220 total_counts=4250.0
    idx=4543 total_counts=672.0
  AR:
    idx=9406 total_counts=631.0
    idx=15730 total_counts=225.0
  ARG1:
    idx=5437 total_counts=740.0
    idx=7371 total_counts=58.0
  ARID3C:
    idx=1841 total_counts=127.0
    idx=1869 total_counts=488.0
  ASIP:
    idx=14508 total_counts=46.0
    idx=12141 total_counts=6949.0
  BNC1:
    idx=15519 total_counts=71908.0
    idx=8109 total_counts=3.0
  BRCC3:
    idx=17267 total_counts=1872.0
    idx=12214 total_counts=822.0
  BRF1:
    idx=1734 total_counts=331.0
    idx=1757 

In [116]:
filtered_nmr_datas[0]

AnnData object with n_obs × n_vars = 24378 × 17303
    obs: 'species', 'sample', 'tissue', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'log1p_total_counts_hb', 'pct_counts_hb', 'total_counts_tf', 'log1p_total_counts_tf', 'pct_counts_tf', 'n_counts', 'qc_composite', 'qc_label', 'sample_id', 'replicate'
    var: 'mt', 'ribo', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'n_cells', 'global_name', 'original_names'

### only choose important columns to keep from obs

In [117]:
filtered_nmr_datas[0].obs[['species', 'sample', 'tissue', 'n_genes_by_counts',
       'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts',
       'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes',
       'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes',
       'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt',
       'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo',
       'total_counts_hb', 'log1p_total_counts_hb', 'pct_counts_hb',
       'total_counts_tf', 'log1p_total_counts_tf', 'pct_counts_tf', 'n_counts',
       'qc_composite', 'qc_label', 'sample_id', 'replicate']]

,species,sample,tissue,n_genes_by_counts,log1p_n_genes_by_counts,total_counts,log1p_total_counts,pct_counts_in_top_50_genes,pct_counts_in_top_100_genes,pct_counts_in_top_200_genes,...,log1p_total_counts_hb,pct_counts_hb,total_counts_tf,log1p_total_counts_tf,pct_counts_tf,n_counts,qc_composite,qc_label,sample_id,replicate
AAACCCAAGAAGCGAA-1-NMR1,nmr,NMR1,cerebral cortex,1559,7.352441,2298.0,7.740230,14.882507,22.062663,32.245431,...,0.0,0.0,23.0,3.178054,1.000870,2298.0,8.0,acceptable,1,1
AAACCCAAGGAGGCAG-1-NMR1,nmr,NMR1,cerebral cortex,1312,7.180070,1777.0,7.483244,14.350028,21.890827,33.145751,...,0.0,0.0,24.0,3.218876,1.350591,1777.0,60.0,acceptable,1,1
AAACCCACAAATCAAG-1-NMR1,nmr,NMR1,cerebral cortex,491,6.198479,679.0,6.522093,29.602356,42.415317,57.142857,...,0.0,0.0,2.0,1.098612,0.294551,679.0,92.0,acceptable,1,1
AAACCCACAGTCTACA-1-NMR1,nmr,NMR1,cerebral cortex,1841,7.518607,3369.0,8.122668,19.709112,27.634313,38.171564,...,0.0,0.0,36.0,3.610918,1.068566,3369.0,1.0,acceptable,1,1
AAACCCAGTACGTAGG-1-NMR1,nmr,NMR1,cerebral cortex,478,6.171701,710.0,6.566672,32.535211,46.760563,60.845070,...,0.0,0.0,8.0,2.197225,1.126761,710.0,92.0,acceptable,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TTTGTTGCACTGAATC-1-NMR2,nmr,NMR2,cerebral cortex,189,5.247024,282.0,5.645447,48.936170,68.439716,100.000000,...,0.0,0.0,8.0,2.197225,2.836879,282.0,25.5,acceptable,2,2
TTTGTTGGTCCCTCAT-1-NMR2,nmr,NMR2,cerebral cortex,1719,7.450080,3030.0,8.016648,19.042904,27.689769,38.184818,...,0.0,0.0,33.0,3.526361,1.089109,3030.0,59.0,acceptable,2,2
TTTGTTGGTTCGGCTG-1-NMR2,nmr,NMR2,cerebral cortex,243,5.497168,447.0,6.104793,52.796421,68.008949,90.380313,...,0.0,0.0,3.0,1.386294,0.671141,447.0,59.0,acceptable,2,2
TTTGTTGGTTCTCGCT-1-NMR2,nmr,NMR2,cerebral cortex,773,6.651572,1431.0,7.266827,28.721174,40.880503,56.254368,...,0.0,0.0,29.0,3.401197,2.026555,1431.0,27.5,acceptable,2,2
